In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
df = pd.read_csv(
    "../data/processed/regression_dataset.csv"
)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["delivery_days"].describe()

In [ ]:
X = df.drop(
    columns=["delivery_days"]
)

y = df["delivery_days"]

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
id_columns = [
    "order_id",
    "customer_id"
]

X = X.drop(
    columns=id_columns,
    errors="ignore"
)

In [ ]:
X.columns.tolist()

In [ ]:
date_columns = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

X = X.drop(
    columns=date_columns,
    errors="ignore"
)

In [ ]:
leakage_columns = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "delivery_delay_days",
    "late_delivery"
]

X = X.drop(
    columns=leakage_columns,
    errors="ignore"
)

In [ ]:
X.columns.tolist()

In [ ]:
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

In [ ]:
print("Numeric:", numeric_features)
print()
print("Categorical:", categorical_features)

In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

In [ ]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_pipeline,
            numeric_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
print(X_train.shape)
print(X_test.shape)

In [ ]:
baseline_prediction = np.repeat(
    y_train.mean(),
    len(y_test)
)

In [ ]:
baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_prediction
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_prediction
)

In [ ]:
print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

In [ ]:
linear_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            LinearRegression()
        )
    ]
)

In [ ]:
linear_model.fit(
    X_train,
    y_train
)

In [ ]:
linear_pred = linear_model.predict(
    X_test
)

In [ ]:
def evaluate_model(
    model_name,
    y_true,
    predictions
):
    mae = mean_absolute_error(
        y_true,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            predictions
        )
    )

    r2 = r2_score(
        y_true,
        predictions
    )

    print(f"\n{model_name}")
    print("-" * 30)
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", r2)

    return {
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }

In [ ]:
linear_result = evaluate_model(
    "Linear Regression",
    y_test,
    linear_pred
)

In [ ]:
rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            RandomForestRegressor(
                n_estimators=200,
                max_depth=15,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [ ]:
rf_model.fit(
    X_train,
    y_train
)

In [ ]:
rf_pred = rf_model.predict(
    X_test
)

In [ ]:
rf_result = evaluate_model(
    "Random Forest",
    y_test,
    rf_pred
)

In [ ]:
gb_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=150,
                learning_rate=0.05,
                max_depth=5,
                random_state=42
            )
        )
    ]
)

In [ ]:
gb_model.fit(
    X_train,
    y_train
)

In [ ]:
gb_pred = gb_model.predict(
    X_test
)

In [ ]:
gb_result = evaluate_model(
    "Gradient Boosting",
    y_test,
    gb_pred
)

In [ ]:
results = pd.DataFrame([
    baseline_result if "baseline_result" in globals() else {
        "Model": "Baseline",
        "MAE": baseline_mae,
        "RMSE": baseline_rmse,
        "R2": baseline_r2
    },
    linear_result,
    rf_result,
    gb_result
])

results

In [ ]:
results.sort_values(
    "MAE"
)

In [ ]:
best_model = gb_model
best_pred = gb_pred

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    y_test,
    best_pred,
    alpha=0.4
)

plt.xlabel("Actual Delivery Days")
plt.ylabel("Predicted Delivery Days")
plt.title("Actual vs Predicted Delivery Time")

plt.show()

In [ ]:
residuals = y_test - best_pred

In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    residuals,
    bins=40
)

plt.title("Residual Distribution")
plt.xlabel("Actual - Predicted")
plt.ylabel("Frequency")

plt.show()

In [ ]:
error_analysis = pd.DataFrame({
    "actual": y_test,
    "predicted": best_pred
})

error_analysis["absolute_error"] = (
    abs(
        error_analysis["actual"]
        -
        error_analysis["predicted"]
    )
)

In [ ]:
error_analysis.sort_values(
    "absolute_error",
    ascending=False
).head(20)

In [ ]:
import joblib

In [ ]:
joblib.dump(
    best_model,
    "../models/delivery_time_model.pkl"
)

In [ ]:
loaded_model = joblib.load(
    "../models/delivery_time_model.pkl"
)

In [ ]:
test_prediction = loaded_model.predict(
    X_test.head(5)
)

test_prediction